In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
from pinecone.exceptions import PineconeException
import fitz  
import pdfplumber 

# ===== Configuration =====
PINECONE_API_KEY = "pcsk_4DqA35_BYaWgQCoVUkGTNDRYenf3NQzwzZX6C685nC2fwMj5qXgnpMXmUcH1eVXRjVfMgw"
PINECONE_ENVIRONMENT = "us-east-1"  # Match your index's region
PINECONE_INDEX_NAME = "quickstart-new-test-pdf-rows-final"
VECTOR_DIM = 384  # For 'all-MiniLM-L6-v2'
PDF_FOLDER_PATH  = "../data/cleaned"  # Path to your pdf

# ===== Pinecone Setup =====
def init_pinecone():
    pc = Pinecone(api_key=PINECONE_API_KEY)

    if PINECONE_INDEX_NAME not in pc.list_indexes().names():
        print(f" Creating index '{PINECONE_INDEX_NAME}'...")
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=VECTOR_DIM,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region=PINECONE_ENVIRONMENT)
        )
    else:
        print(f" Index '{PINECONE_INDEX_NAME}' already exists.")

    return pc.Index(PINECONE_INDEX_NAME)

# ===== PDF Text Extraction and Processing =====
def read_pdf_texts(pdf_folder):
    rows = []
    for file in os.listdir(pdf_folder):
        if file.endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, file)
            print(f"\n--- Reading file: {file} ---")
            row_count = 0
            table_count = 0
            text_count = 0

            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages):
                    table = page.extract_table()
                    if table:
                        table_count += 1
                        for row in table[1:]:  # Skip header
                            row_text = " | ".join(str(cell) for cell in row)
                            rows.append({
                                "text": f"{file} | Page {page_num+1}: {row_text}",
                                "file": file,
                                "page": page_num + 1,
                                "type": "table"
                            })
                            row_count += 1
                    else:
                        text = page.extract_text()
                        if text:
                            for line in text.strip().split("\n"):
                                if line.strip():
                                    rows.append({
                                        "text": f"{file} | Page {page_num+1}: {line.strip()}",
                                        "file": file,
                                        "page": page_num + 1,
                                        "type": "text"
                                    })
                                    row_count += 1
                                    text_count += 1
            print(f"Extracted {row_count} rows (tables: {table_count}, text lines: {text_count})")
    return rows

# ===== Embedding and Upserting to Pinecone =====
def upsert_vectors(index, rows, batch_size=100):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    print("Encoding and upserting vectors in batches...")

    for i in range(0, len(rows), batch_size):
        batch_rows = rows[i:i + batch_size]
        texts = [row["text"] for row in batch_rows]
        embeddings = model.encode(texts, show_progress_bar=True)

        vectors = []
        for j, (emb, row) in enumerate(zip(embeddings, batch_rows)):
            vector_id = f"{row['file']}|{row['page']}|{row['type']}-{i+j}"
            vectors.append((vector_id, emb.tolist(), row))
            print(f"Prepared ID: {vector_id}")

        index.upsert(vectors=vectors)
        print(f"Upserted batch {i} - {i + len(batch_rows) - 1} | Total: {len(vectors)} vectors")

# ===== Confirm Vector Count =====
def check_vector_count(index):
    stats = index.describe_index_stats()
    total_vectors = stats.get('total_vector_count', 0)
    print(f"\nTotal vectors in index: {total_vectors}")

# ===== Main =====
def main():
    try:
        print("Initializing Pinecone...")
        index = init_pinecone()

        print("\nReading PDF files...")
        rows = read_pdf_texts(PDF_FOLDER_PATH)

        print("\nGenerating and upserting embeddings...")
        upsert_vectors(index, rows)

        print("\nChecking index stats...")
        check_vector_count(index)

    except PineconeException as e:
        print(f"Pinecone error: {str(e)}")
    except Exception as e:
        print(f"General error: {str(e)}")

if __name__ == "__main__":
    main()

In [ ]:
# insert_vectors.py

import os
import sys
import pandas as pd
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

# ===== Configuration =====
PINECONE_API_KEY = "pcsk_4DqA35_BYaWgQCoVUkGTNDRYenf3NQzwzZX6C685nC2fwMj5qXgnpMXmUcH1eVXRjVfMgw"
PINECONE_ENVIRONMENT = "gcp-starter"  # Format: cloud-region
PINECONE_INDEX_NAME = "my-index"
VECTOR_DIM = 384  # Must match the embedding model
CSV_PATH = "../data/cleaned/bpx_i_cleaned.csv"  # Path to your CSV file

# ===== Pinecone Setup =====
def init_pinecone():
    cloud, region = PINECONE_ENVIRONMENT.split("-")
    pc = Pinecone(api_key=PINECONE_API_KEY)

    if PINECONE_INDEX_NAME not in pc.list_indexes().names():
        print(f"Creating index '{PINECONE_INDEX_NAME}'...")
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=VECTOR_DIM,
            metric="cosine",
            spec=ServerlessSpec(cloud=cloud, region=region)
        )
    else:
        print(f"Index '{PINECONE_INDEX_NAME}' already exists.")

    return pc.Index(PINECONE_INDEX_NAME)

def upsert_vectors(index, vectors):
    index.upsert(vectors=vectors)

# ===== CSV Processing =====
def read_csv_rows(csv_path):
    df = pd.read_csv(csv_path).fillna("")
    rows = df.astype(str).apply(lambda row: " | ".join(row), axis=1).tolist()
    return rows

# ===== Main Flow =====
def main():
    print("Initializing Pinecone...")
    index = init_pinecone()

    print("Loading embedding model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim

    print("Reading CSV and generating embeddings...")
    rows = read_csv_rows(CSV_PATH)
    embeddings = model.encode(rows).tolist()

    print("Inserting vectors to Pinecone...")
    vectors = [("row-" + str(i), emb) for i, emb in enumerate(embeddings)]
    upsert_vectors(index, vectors)

    print(f" Successfully inserted {len(vectors)} vectors into Pinecone.")

if __name__ == "__main__":
    main()
